# PySpark - Date & Time Operations
This notebook demonstrates how to perform common date manipulations using PySpark's built-in functions.
We'll explore converting dates, formatting, adding intervals, and computing differences.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Initialize Spark session
spark = SparkSession.builder \
    .appName("DateOperations") \
    .getOrCreate()

print("Spark session started successfully")

## Part 1 — Basic Date Conversions and Formatting
Here we take raw date strings in different formats and apply various transformation functions.

In [ ]:
# Sample booking records with two different date formats
booking_data = [
    ("2025-09-10", "12/09/2025"),
    ("2025-11-22", "25/11/2025")
]

# Create DataFrame with column names
bookings_df = spark.createDataFrame(booking_data, ["check_in", "check_out"])

bookings_df.show()

In [ ]:
# Apply date transformations
transformed_df = bookings_df.select(

    # System's current date
    F.current_date().alias("Today"),

    # Parse check_in from default ISO format (yyyy-MM-dd)
    F.to_date(F.col("check_in")).alias("CheckIn_Parsed"),

    # Parse check_out using explicit format dd/MM/yyyy
    F.to_date(F.col("check_out"), "dd/MM/yyyy").alias("CheckOut_Parsed"),

    # Reformat check_in date as DD-MM-YYYY string
    F.date_format(F.col("check_in"), "dd-MM-yyyy").alias("Formatted_CheckIn"),

    # Add 3 months to check_in (e.g., for subscription renewal)
    F.add_months(F.col("check_in"), 3).alias("Renewal_Date"),

    # Add 15 days to check_in
    F.date_add(F.col("check_in"), 15).alias("Extended_CheckIn"),

    # Number of days between check_out and check_in
    F.datediff(
        F.to_date(F.col("check_out"), "dd/MM/yyyy"),
        F.to_date(F.col("check_in"))
    ).alias("Stay_Duration_Days")
)

transformed_df.show(truncate=False)

## Common Date Format Patterns in PySpark

| Pattern | Description | Example |
| --- | --- | --- |
| `yyyy-MM-dd` | ISO standard (Year-Month-Day) | 2025-09-10 |
| `dd/MM/yyyy` | Day/Month/Year with slashes | 10/09/2025 |
| `MM-dd-yyyy` | US format (Month-Day-Year) | 09-10-2025 |
| `dd-MM-yyyy` | Day-Month-Year with dashes | 10-09-2025 |
| `dd MMM yyyy` | Day Abbrev-Month Year | 10 Sep 2025 |

## Part 2 — Employee Tenure Analysis using Advanced Date Functions
Using a sample HR dataset, we'll extract detailed date components and compute tenure metrics.

In [ ]:
# HR records — employee ID, name, and date of joining
hr_data = [
    (201, "Riya",   "2021-06-18"),
    (202, "Arjun",  "2022-09-05"),
    (203, "Simran", "2020-12-30"),
    (204, "Karan",  "2023-04-14")
]

hr_df = spark.createDataFrame(hr_data, ["Emp_ID", "Name", "DOJ"])

# Convert DOJ from string to DateType
hr_df = hr_df.withColumn(
    "DOJ",
    F.to_date(F.col("DOJ"), "yyyy-MM-dd")
)

hr_df.printSchema()

In [ ]:
# Apply advanced date functions for HR analytics
hr_analysis = hr_df.select(

    F.col("Emp_ID"),
    F.col("Name"),
    F.col("DOJ"),

    # Total months since joining (rounded to 2 decimal places)
    F.round(
        F.months_between(F.current_date(), F.col("DOJ")), 2
    ).alias("Tenure_Months"),

    # First Monday after joining date
    F.next_day(F.col("DOJ"), "Mon").alias("First_Monday"),

    # Start of the year when employee joined
    F.trunc(F.col("DOJ"), "year").alias("Year_Start"),

    # Start of the month when employee joined
    F.trunc(F.col("DOJ"), "month").alias("Month_Start"),

    # Calendar year of joining
    F.year(F.col("DOJ")).alias("Year_Joined"),

    # Business quarter of joining (Q1-Q4)
    F.quarter(F.col("DOJ")).alias("Quarter"),

    # Month number (1 = Jan, 12 = Dec)
    F.month(F.col("DOJ")).alias("Month_No"),

    # Day of week (1 = Sunday, 7 = Saturday)
    F.dayofweek(F.col("DOJ")).alias("Weekday_No")
)

hr_analysis.show(truncate=False)

## Summary

| Function | Purpose |
| --- | --- |
| `current_date()` | Returns today's system date |
| `to_date(col, format)` | Converts string to DateType |
| `date_format(col, fmt)` | Converts date to formatted string |
| `add_months(col, n)` | Adds n months to a date |
| `date_add(col, n)` | Adds n days to a date |
| `datediff(end, start)` | Days between two dates |
| `months_between(d1, d2)` | Months between two dates (decimal) |
| `next_day(col, weekday)` | Nearest future given weekday |
| `trunc(col, unit)` | Truncates to start of year/month |
| `year / month / quarter / dayofweek` | Extract individual date parts |